## Import Prerequis

In [1]:
import sys
import os
import subprocess
import ctypes
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Configuration des chemins vers la racine du projet
sys.path.append(os.path.abspath(os.path.join('..')))

### Compile Library C and generate JSON (make)

In [2]:
try:
    result = subprocess.run(
        "make -C ../libc clean && make -C ../libc",
        shell=True,
        capture_output=True,
        text=True
    )
    # print(result.stdout)

    if result.stderr:
        print(result.stderr)
    
    if result.returncode != 0:
        print(f"Build failed with exit code {result.returncode}")
        sys.exit(1)
    else:
        print("Build succeeded.")

except Exception as e:
    print(f"Build failed: {e}")
    sys.exit(1)

Build succeeded.


## Import Loader

In [3]:
from engine.interop.loader import Loader
from engine.interop.linearModel import LinearModel

# Chargement du Singleton Loader
LIB_NAME = "libc"
LIB_FOLDER = "../libc"
BUILD_FOLDER = "../libc/build"
SPECS_FOLDER = "../libc/specs"
DEPENDENCIES_FOLDER = r"C:\msys64\mingw64\bin"

Loader.loadLibrary(LIB_NAME, lib_folder=LIB_FOLDER, build_folder=BUILD_FOLDER, specs_folder=SPECS_FOLDER, dependencies_bin_folder=DEPENDENCIES_FOLDER)
print("✓ Bibliothèque C chargée avec succès !")

✓ Bibliothèque C chargée avec succès !


## Chargement du Dataset Réel (cleaned)

In [4]:
# dataset_csv_path = os.path.join("G:", ".shortcut-targets-by-id", "1OIODRlxR-9dknLaZ6jub08oiAH0E4eGR", "DATASET", "art-genre-dataset", "cleaned_dataset")
dataset_csv_path = os.path.join("..", "dataset")
resolution = "64x64"

# Limite du nombre d'images à traiter par catégorie (-1 pour aucune limite)
limit = 100

In [ ]:
# On définit les catégories et leurs chemins de données et CSV

categories = {
    "impressionism" : {
        "data_folder_path": os.path.join(dataset_csv_path, resolution, "impressionism"),
        "csv_path": os.path.join(dataset_csv_path, "impressionism_clean.csv")
        },
    "realism" : {
        "data_folder_path": os.path.join(dataset_csv_path, resolution, "realism"),
        "csv_path": os.path.join(dataset_csv_path, "realism_clean.csv")
    },
    "romanticism" : {
        "data_folder_path": os.path.join(dataset_csv_path, resolution, "romanticism"),
        "csv_path": os.path.join(dataset_csv_path, "romanticism_clean.csv")
    }
}

df_csv_categories = dict()

df_csv_all_shuffled = {
    "train": pd.DataFrame(),
    "test": pd.DataFrame()
}

for category in categories:
    df = pd.read_csv(categories[category]["csv_path"])

    if df.empty:
        raise ValueError(f"CSV file for category '{category}' is empty or not found at path: {categories[category]['csv_path']}")
    
    if limit > 0:
        df = df.head(limit)

    # Modifier le nom du fichier par son chemin complet
    if "Nom_Fichier" not in df.columns:
        raise ValueError("Column 'Nom_Fichier' does not exist in the DataFrame.")
    df["filepath"] = df["Nom_Fichier"].apply(lambda x: os.path.join(categories[category]["data_folder_path"], x))

    # Ajouter une colonne category (Y)
    for c in categories:
        if c in df.columns:
            raise ValueError(f"Column '{c}' already exists in the DataFrame.")
        df[f"{c}"] = 1 if c == category else -1
    
    # Separation test / train : 70% train, 30% test
    df_train = df.sample(frac=0.7, random_state=42)
    df_test = df.drop(df_train.index)
    
    df_csv_categories[category] = {
        "train": df_train,
        "test": df_test
    }

    if df_csv_all_shuffled["train"].empty:
        df_csv_all_shuffled["train"] = df_train
        df_csv_all_shuffled["test"] = df_test
    else:
        df_csv_all_shuffled["train"] = pd.concat([df_csv_all_shuffled["train"], df_train], ignore_index=True)
        df_csv_all_shuffled["test"] = pd.concat([df_csv_all_shuffled["test"], df_test], ignore_index=True)

# On mélange les données pour éviter des conclusions via l'odre des catégories
df_csv_all_shuffled["train"] = df_csv_all_shuffled["train"].sample(frac=1, random_state=42).reset_index(drop=True)
df_csv_all_shuffled["test"] = df_csv_all_shuffled["test"].sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# On sépare les données d'entrainement et de test en X et Y

df_X_filepaths = {
    "train": df_csv_all_shuffled["train"]["filepath"].tolist(),
    "test": df_csv_all_shuffled["test"]["filepath"].tolist()
}

df_Y = {
    "train": dict(),
    "test": dict()
}

for category in categories:
    df_Y["train"][category] = list(df_csv_all_shuffled["train"][f"{category}"])
    df_Y["test"][category] = list(df_csv_all_shuffled["test"][f"{category}"])

    
    df_csv_all_shuffled["train"].drop(columns=[f"{category}"], inplace=True)
    df_csv_all_shuffled["test"].drop(columns=[f"{category}"], inplace=True)

## Chargement du Dataset Réel (Images 64x64) via Pillow

In [ ]:
df_X = dict()

for step in df_X_filepaths:

    print(f"Loading {step}...")

    df_X[step] = []

    enumerate_df_X_filepaths = list(enumerate(df_X_filepaths[step]))

    for i, filepath in enumerate_df_X_filepaths:

        if i % 50 == 0:
            print(f"{i}/{enumerate_df_X_filepaths[-1][0]}", filepath)

        img = Image.open(filepath).convert("RGB")
        
        # PAS BESOIN DE RESIZE POUR LE DATASET (en revanche il faut bien penser à resize les vrai images hors dataset)
        # img = img.resize((64, 64))

        # On normalise les valeurs des pixels entre -1 et 1 pour le modèle (normalisation par standardisation)
        img_array = (np.array(img).flatten() / 127.5 - 1.0).astype(np.float32)
        df_X[step].append(img_array)

    print(f"{step} terminé")

df_X["train"] = np.concatenate(df_X["train"]).tolist()

Loading train...
0/209 ..\dataset\64x64\impressionism\abdullah-suriosubroto_landscape-of-paddy-field.jpg
50/209 ..\dataset\64x64\impressionism\adam-baltatu_summer-sky.jpg
100/209 ..\dataset\64x64\realism\alekos-kontopoulos_portrait-of-eugenia-wife-of-kleomenis-tsitsaras.jpg
150/209 ..\dataset\64x64\impressionism\abdullah-suriosubroto_mountain-view.jpg
200/209 ..\dataset\64x64\realism\adolf-hitler_flowers-with-roots.jpg
train terminé
Loading test...
0/89 ..\dataset\64x64\realism\aleksey-savrasov_autumn-sokolniki.jpg
50/89 ..\dataset\64x64\impressionism\adam-baltatu_house-on-siret-valley.jpg
test terminé


## Train

In [ ]:
models = dict()

for category in categories:
    print(f"Training model for category: {category}")

    W_length = (len(df_X["train"]) // len(df_Y["train"][category])) # le biais est géré dans la lib !!!

    models[category] = LinearModel.init_random(input_dim=W_length)
    models[category].train(
        dataset_inputs=df_X["train"],
        dataset_expected_outputs=df_Y["train"][category],
        is_classification=True,
        alpha=0.01,
        epochs=1000
    )
    print(f"Model for category '{category}' trained successfully.\n")

Training model for category: impressionism
Model for category 'impressionism' trained successfully.

Training model for category: realism
Model for category 'realism' trained successfully.

Training model for category: romanticism
Model for category 'romanticism' trained successfully.



## Test

In [9]:
predictions = dict()

for category in categories:
    print(f"Evaluating model for category: {category}")
    predictions[category] = models[category].predict(df_X["test"][0], is_classification=False)

Evaluating model for category: impressionism
Evaluating model for category: realism
Evaluating model for category: romanticism


In [10]:
for category in categories:
    expected = df_Y["test"][category][0]
    predicted = predictions[category]

    print(f"Category: {category}")
    print(f"Expected: {expected}, Predicted: {predicted}\n")

Category: impressionism
Expected: False, Predicted: 33.525943756103516

Category: realism
Expected: True, Predicted: 34.269683837890625

Category: romanticism
Expected: False, Predicted: -14.684183120727539

